# Sonata + VoxelGroupClassifier: Full Pipeline on Colab
## Entropy-Guided Adaptive Masking for 3D Scene Pretraining

**Requirements**: Colab with T4/V100/A100 GPU (Pro/Pro+ recommended).

**Steps**: Clone → Install → Convert Data → Train → Monitor

> ⚠️ After installing spconv, Colab will ask you to **restart the runtime**.
> Click "Restart Runtime", then continue from the same cell.

In [ ]:
# ═══════════════════════════════════════════════
# CONFIGURATION — change these before running
# ═══════════════════════════════════════════════
EPOCHS = 10          # training epochs
MASKING_MODE = "cosine"  # "random" | "cosine" | "vgc"
GPU_COUNT = 1        # number of GPUs to use

In [30]:
# ═══════════════════════════════════════
# CELL 1 — Base Environment (Final Stable)
# ═══════════════════════════════════════
import os, sys, shutil

# 1. Install missing runtime dependencies
!pip install -q addict pyyaml h5py tensorboard timm torch-scatter ninja ccimport pccm pyecharts

# 2. Handle tensorview
try:
    import tensorview
except:
    !pip install -q "tensorview==0.3.0"
    import tensorview

tv_include = os.path.join(os.path.dirname(tensorview.__file__), "include")
os.environ["CPATH"] = tv_include + ":" + os.environ.get("CPATH", "")

# 3. Install spconv
!pip install -q cumm-cu118 spconv-cu118

# 4. Fresh Repo Setup (ensure no missing files)
%cd /content
if os.path.exists("voxel_group_classifier"):
    shutil.rmtree("voxel_group_classifier")
!git clone https://github.com/yuhang-wang-xjtu/voxel_group_classifier.git

%cd voxel_group_classifier
print("Environment Base Ready with Fresh Clone.")

/content
Cloning into 'voxel_group_classifier'...
remote: Enumerating objects: 725, done.
remote: Counting objects: 100% (725/725), done.
remote: Compressing objects: 100% (373/373), done.
remote: Total 725 (delta 340), reused 715 (delta 336), pack-reused 0 (from 0)
Receiving objects: 100% (725/725), 2.12 MiB | 6.02 MiB/s, done.
Resolving deltas: 100% (340/340), done.
/content/voxel_group_classifier
Environment Base Ready with Fresh Clone.


In [16]:
# ═══════════════════════════════════════
# CELL 1.5 — Incremental Operator Build (Final Fix with Dependencies)
# ═══════════════════════════════════════
import os, sys, shutil, importlib.util

def is_installed(name):
    try:
        return importlib.util.find_spec(name) is not None
    except:
        return False

# 1. Install system-level dependencies for pointgroup_ops
print("--- Installing system dependencies (sparsehash) ---")
!apt-get install -y -qq libsparsehash-dev

# Force CUDA environment variables
os.environ["CUDA_HOME"] = "/usr/local/cuda"
os.environ["PATH"] = f"{os.environ['CUDA_HOME']}/bin:{os.environ['PATH']}"

libs_to_build = {
    "libs/pointops": "pointops",
    "libs/pointops2": "pointops2",
    "libs/pointgroup_ops": "pointgroup_ops"
}

base_dir = "/content/voxel_group_classifier"
%cd {base_dir}

for folder, pkg in libs_to_build.items():
    if is_installed(pkg):
        print(f"✅ {pkg} is already installed. Skipping.")
        continue

    full_path = os.path.join(base_dir, folder)
    if os.path.exists(full_path):
        print(f"\n--- Compiling {folder} ---")
        %cd {full_path}

        # Force a fresh start for the compiler
        for d in ["build", f"{pkg}.egg-info", "dist"]:
            if os.path.exists(d): shutil.rmtree(d)

        # Try installing using pip which handles build dependencies better
        # Note: --no-build-isolation is used because we pre-installed torch/spconv
        res = os.system("MAX_JOBS=2 pip install . --no-build-isolation")

        if res == 0:
            print(f"Successfully installed {pkg}!")
        else:
            print(f"⚠️ Pip failed for {pkg}, trying legacy setup.py...")
            !python setup.py build_ext --inplace
            !python setup.py install

        %cd {base_dir}
    else:
        print(f"❌ Error: {folder} not found!")

print("\nOperator check complete.")

--- Installing system dependencies (sparsehash) ---
Selecting previously unselected package libsparsehash-dev.
(Reading database ... 122403 files and directories currently installed.)
Preparing to unpack .../libsparsehash-dev_2.0.3-2_all.deb ...
Unpacking libsparsehash-dev (2.0.3-2) ...
Setting up libsparsehash-dev (2.0.3-2) ...
/content/voxel_group_classifier
✅ pointops is already installed. Skipping.
✅ pointops2 is already installed. Skipping.

--- Compiling libs/pointgroup_ops ---
/content/voxel_group_classifier/libs/pointgroup_ops
/content/voxel_group_classifier

Operator check complete.


In [17]:
# ═══════════════════════════════════════
# CELL 2 — Data: HDF5 → Pointcept (3 min)
# ═══════════════════════════════════════
import numpy as np, h5py, glob

# Download HDF5 from HuggingFace mirror (~1.7 GB)
HDF5_URL = "https://huggingface.co/datasets/cminst/S3DIS/resolve/main/indoor3d_sem_seg_hdf5_data.zip"
HDF5_DIR = "./indoor3d_sem_seg_hdf5_data"
OUT_ROOT = "./data/s3dis"

if not os.path.isdir(HDF5_DIR):
    print(f"Downloading S3DIS HDF5 from HuggingFace (~1.7 GB)...")
    import urllib.request, zipfile
    urllib.request.urlretrieve(HDF5_URL, "/tmp/s3dis.zip")
    with zipfile.ZipFile("/tmp/s3dis.zip") as zf:
        zf.extractall(HDF5_DIR)
    print("Download complete.")
else:
    print("Already downloaded.")

# Convert HDF5 → Pointcept format (merge blocks per HDF5 file into one room)
h5_files = sorted(glob.glob(f"{HDF5_DIR}/*/*.h5"))
print(f"Converting {len(h5_files)} HDF5 files to Pointcept format...")

written = 0
for h5_path in h5_files:
    fname = os.path.basename(h5_path).replace(".h5", "")
    area_idx = int(fname.split("_")[-1])
    area = f"Area_{area_idx}"

    with h5py.File(h5_path, "r") as f:
        data = f["data"][:]
        labels = f["label"][:]

    xyz = data[..., :3].reshape(-1, 3).astype(np.float32)
    rgb = data[..., 3:6].reshape(-1, 3).astype(np.uint8)
    seg = labels.reshape(-1, 1).astype(np.int16)
    ins = np.zeros_like(seg, dtype=np.int16)
    if data.shape[-1] >= 9:
        nrm = data[..., 6:9].reshape(-1, 3).astype(np.float32)
    else:
        nrm = None

    room_dir = os.path.join(OUT_ROOT, area, fname)
    os.makedirs(room_dir, exist_ok=True)
    np.save(os.path.join(room_dir, "coord.npy"), xyz)
    np.save(os.path.join(room_dir, "color.npy"), rgb)
    np.save(os.path.join(room_dir, "segment.npy"), seg)
    np.save(os.path.join(room_dir, "instance.npy"), ins)
    if nrm is not None:
        np.save(os.path.join(room_dir, "normal.npy"), nrm)
    written += 1

print(f"Done. {written} rooms written to {OUT_ROOT}/")
!ls -d ./data/s3dis/Area_* | head -10

Download complete.
Converting 24 HDF5 files to Pointcept format...
Done. 24 rooms written to ./data/s3dis/
./data/s3dis/Area_0
./data/s3dis/Area_1
./data/s3dis/Area_10
./data/s3dis/Area_11
./data/s3dis/Area_12
./data/s3dis/Area_13
./data/s3dis/Area_14
./data/s3dis/Area_15
./data/s3dis/Area_16
./data/s3dis/Area_17


In [36]:
# ═══════════════════════════════════════
# CELL 3 — Train Sonata (10 epoch smoke test)
# ═══════════════════════════════════════

import os, sys

# 1. Install dependencies
!pip install -q addict pyyaml yapf tensorboardX

# 2. Force Package Registration via Temporary setup.py
PROJECT_ROOT = "/content/voxel_group_classifier"
os.chdir(PROJECT_ROOT)

# Ensure all directories in pointcept have __init__.py
!find pointcept -type d -not -path '*/.*' -exec touch {}/__init__.py \;

# Create a robust setup.py
setup_content = """
from setuptools import setup, find_packages
setup(
    name='pointcept',
    version='0.1',
    packages=find_packages(),
    zip_safe=False,
)
"""
with open("setup.py", "w") as f:
    f.write(setup_content)

# Re-install in editable mode
!pip install -e .

# 3. Config selection
if 'MASKING_MODE' not in globals(): MASKING_MODE = "cosine"
if 'GPU_COUNT' not in globals(): GPU_COUNT = 1
if 'EPOCHS' not in globals(): EPOCHS = 10

config_file = f"configs/sonata/pretrain-sonata-v1m2-{MASKING_MODE}-smoketest.py"
if not os.path.exists(config_file):
    config_file = "configs/sonata/pretrain-sonata-v1m2-cosine-smoketest.py"

save_path = f"exp/{MASKING_MODE}_{GPU_COUNT}GPU_{EPOCHS}ep"
common_opts = f"save_path={save_path} epoch={EPOCHS} data.train.datasets.0.data_root=data/s3dis"

# 4. Launch
print(f"Launching training with config: {config_file}")
!python tools/train.py --config-file {config_file} --options {common_opts} 2>&1

Obtaining file:///content/voxel_group_classifier
  Preparing metadata (setup.py) ... done
  Attempting uninstall: pointcept
    Found existing installation: pointcept 0.1
    Uninstalling pointcept-0.1:
      Successfully uninstalled pointcept-0.1
  Running setup.py develop for pointcept
Launching training with config: configs/sonata/pretrain-sonata-v1m2-cosine-smoketest.py
Traceback (most recent call last):
  File "/content/voxel_group_classifier/tools/train.py", line 13, in <module>
    from pointcept.engines.train import TRAINERS
  File "/content/voxel_group_classifier/pointcept/engines/train.py", line 29, in <module>
    from pointcept.datasets import build_dataset, point_collate_fn, collate_fn
  File "/content/voxel_group_classifier/pointcept/datasets/__init__.py", line 1, in <module>
    from .defaults import (
  File "/content/voxel_group_classifier/pointcept/datasets/defaults.py", line 20, in <module>
    from pointcept.utils.logger import get_root_logger
ModuleNotFoundError: N

In [ ]:
# ═══════════════════════════════
# CELL 4 — Monitor Results
# ═══════════════════════════════

# TensorBoard
%load_ext tensorboard
%tensorboard --logdir exp --port 6006

# Or view training log
# !tail -50 exp/vgc_*/train.log

## Troubleshooting

**spconv installation fails**: Try `!pip install spconv-cu121` (CUDA 12.1).

**CUDA extension build fails**: Colab may have unexpected CUDA version. Run `!nvcc --version` to check, match spconv version accordingly.

**Out of memory**: Reduce batch_size in config or use 1 GPU.

**Data loader error**: Check `!ls data/s3dis/Area_1/` — should show room directories.